# VoxSync Colab (online)

Transcribe video o YouTube, traduce a espanol y genera SRT.

1. Entorno de ejecucion > Cambiar tipo de entorno > GPU (T4)
2. Ejecuta las celdas en orden
3. Al final descarga el .srt


## 1. Instalar dependencias


In [ ]:
!pip install -q openai-whisper yt-dlp deep-translator
!apt -y -qq install ffmpeg
print("OK")


## 2. Configuracion


In [ ]:
YOUTUBE_URL = ""  # ejemplo: "https://www.youtube.com/watch?v=XXXX"
SOURCE_LANG = "en"
TARGET_LANG = "es"
WHISPER_MODEL = "small"  # tiny | base | small | medium | large
print("Config OK")


## 3. Cargar video


In [ ]:
from pathlib import Path
import subprocess

work = Path("/content/voxsync")
work.mkdir(exist_ok=True)

if YOUTUBE_URL.strip():
    print("Descargando YouTube...")
    subprocess.run([
        "yt-dlp", "-f", "bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]/b",
        "-o", str(work / "%(title)s.%(ext)s"), "--no-playlist", YOUTUBE_URL.strip()
    ], check=True)
    cands = []
    for ext in (".mp4", ".webm", ".mkv", ".m4a", ".mp3"):
        cands.extend(work.glob("*" + ext))
    video_path = max(cands, key=lambda p: p.stat().st_mtime)
    print("Video:", video_path.name)
else:
    from google.colab import files
    print("Sube un video o audio...")
    uploaded = files.upload()
    name = list(uploaded.keys())[0]
    video_path = work / name
    video_path.write_bytes(uploaded[name])
    print("Archivo:", video_path.name)

print("Listo:", video_path)


## 4. Extraer audio


In [ ]:
audio_path = work / "audio.wav"
subprocess.run([
    "ffmpeg", "-y", "-i", str(video_path),
    "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1",
    str(audio_path)
], check=True, capture_output=True)
print("Audio OK:", audio_path)


## 5. Transcribir


In [ ]:
import whisper
print("Cargando", WHISPER_MODEL)
model = whisper.load_model(WHISPER_MODEL)
print("Transcribiendo...")
result = model.transcribe(str(audio_path), language=SOURCE_LANG or None, verbose=False)
segments = result["segments"]
print(len(segments), "segmentos")
for s in segments[:3]:
    print(round(s["start"], 1), s["text"][:100])


## 6. Traducir


In [ ]:
from deep_translator import GoogleTranslator
translator = GoogleTranslator(source=SOURCE_LANG or "auto", target=TARGET_LANG)

def tr(text):
    text = (text or "").strip()
    if not text or SOURCE_LANG == TARGET_LANG:
        return text
    try:
        return translator.translate(text)
    except Exception as e:
        print("aviso:", e)
        return text

print("Traduciendo...")
for i, seg in enumerate(segments):
    seg["translation"] = tr(seg["text"])
    if (i + 1) % 20 == 0:
        print(i + 1, "/", len(segments))
print("OK")
for s in segments[:3]:
    print("ORIG:", s["text"][:80])
    print("TRAD:", s["translation"][:80])
    print()


## 7. Generar SRT


In [ ]:
def ts(seconds):
    total = int(float(seconds))
    ms = int((float(seconds) - total) * 1000)
    h, rem = divmod(total, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

base = video_path.stem
srt_path = work / f"{base}.{TARGET_LANG}.srt"
srt_bi = work / f"{base}.bilingual.srt"

with open(srt_path, "w", encoding="utf-8") as f:
    for i, seg in enumerate(segments, 1):
        text = (seg.get("translation") or seg["text"]).strip()
        f.write(f"{i}\n{ts(seg['start'])} --> {ts(seg['end'])}\n{text}\n\n")

with open(srt_bi, "w", encoding="utf-8") as f:
    for i, seg in enumerate(segments, 1):
        orig = seg["text"].strip()
        trans = (seg.get("translation") or orig).strip()
        body = orig + "\n" + trans if trans != orig else orig
        f.write(f"{i}\n{ts(seg['start'])} --> {ts(seg['end'])}\n{body}\n\n")

print("SRT:", srt_path)
print("Bilingual:", srt_bi)


## 8. Descargar


In [ ]:
from google.colab import files
files.download(str(srt_path))
files.download(str(srt_bi))
print("Listo - mira tus descargas")
